## AI Meeting Minutes Generator Project

In [ ]:
#Imports

import os
import gc
import torch
from IPython.display import display, Markdown, Audio, Image
from google.colab import userdata, drive
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TextStreamer,
    BitsAndBytesConfig,
    pipeline
)

from diffusers import AutoPipelineForText2Image


In [ ]:
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
hf_token = userdata.get("HF_TOKEN")

if hf_token and hf_token.startswith('hf_'):
    login(hf_token, add_to_git_credential=True)
    print("✅ Logged in to HuggingFace")
else:
    print("❌ HF_TOKEN not found. Add it via the key icon in the left sidebar.")

In [ ]:
# Model Registry

# Model sizes and memory needs:
#   Gemma  270M params  → ~0.5GB  → no quantization needed
#   Llama  1B params    → ~2GB    → quantize to ~0.5GB
#   DeepSeek 1.5B       → ~3GB    → quantize to ~0.8GB (reasoning model)
#   Phi-4 mini          → ~3.8B   → quantize to ~2GB
#   Qwen   4B params    → ~8GB    → quantize to ~2.5GB

MODELS = {
    "llama":     {"id": "meta-llama/Llama-3.2-1B-Instruct",          "quant": True,  "tokens": 500},
    "phi":       {"id": "microsoft/Phi-4-mini-instruct",              "quant": True,  "tokens": 500},
    "gemma":     {"id": "google/gemma-3-270m-it",                     "quant": False, "tokens": 500},
    "qwen":      {"id": "Qwen/Qwen3-4B-Instruct-2507",               "quant": True,  "tokens": 600},
    "deepseek":  {"id": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B", "quant": False, "tokens": 800},
}


In [ ]:
# quantization config

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
drive.mount("/content/drive")
audio_filename = "/content/drive/MyDrive/llms/denver_extract.mp3"

if os.path.exists(audio_filename):
    size_mb = os.path.getsize(audio_filename) / 1e6
    print(f"✅ Audio file found: {size_mb:.1f} MB")
    display(Audio(audio_filename))  # play the audio in notebook
else:
    print(f"❌ File not found at {audio_filename}")
    print("Please place your audio file at the path above")

In [ ]:
# pipeline() is from the transformers library
# task = "automatic-speech-recognition" → HF looks up its registry and knows
#         Whisper is the standard model family for this task


# return_timestamps=True → also returns when each word was spoken
# dtype=torch.float16   → run inference in fp16 (faster on GPU)

whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model = "openai/whisper-medium",
    dtype = torch.float16,
    device = "cuda",
    return_timestamps = True
)

result = whisper_pipe(audio_filename)
transcription = result["text"]
print(f"Transcription: {transcription}")


In [ ]:
del whisper_pipe
gc.collect()
torch.cuda.empty_cache()

## NER - Named Entity Recognition
### Scans Script token by token and labels it as

## PER/ORG/LOC/MISC:

### *Uses Bio Tagging*

## B - Beginning
## I - Inside
## O - Outside

In [ ]:
ner_pipe = pipeline("ner", device="cuda", aggregation_strategy="simple")
ner_res = ner_pipe(transcription)
print(ner_res)

In [ ]:
#group by entities type
entities = {"PER": [], "ORG": [], "LOC": [], "MISC": []}

for entity in ner_res:
  group = entity["entity_group"]
  word = entity["word"]
  score = entity["score"]
  if group in entities and score > 0.85:
    if word not in entities[group]:
      entities[group].append(word)
print("\n✅ Named Entities Found:")
print(f"  👤 People (PER):        {entities['PER'] or 'none found in first 1000 chars'}")
print(f"  🏢 Organizations (ORG): {entities['ORG'] or 'none found'}")
print(f"  📍 Locations (LOC):     {entities['LOC'] or 'none found'}")
print(f"  🔖 Misc (MISC):         {entities['MISC'] or 'none found'}")



In [ ]:
del ner_pipe
gc.collect()
torch.cuda.empty_cache()

## Sentiment Analysis



In [ ]:
sent_pipe = pipeline("sentiment-analysis", device = "cuda")
short_transcription = " ".join(transcription.split()[:400])
sent_res = sent_pipe(short_transcription)
print(sent_res)
label = sent_res[0]["label"]
score = sent_res[0]["score"]

In [ ]:
del sent_pipe
gc.collect()
torch.cuda.empty_cache()

## Generate Meeting Minutes With LLMs

In [ ]:
system_message = """
You produce minutes of meetings from transcripts.
Write structured minutes in markdown (no code blocks) including:
- Summary with attendees, location, date
- Key discussion points
- Decisions made
- Action items with owners
- Takeaways
Be concise and professional.
"""

In [ ]:
user_prompt = f"""
Below is a transcript of a Denver city council meeting.

Additional context from automated analysis:
- Named entities found: {entities}
- Meeting tone: {label} ({score:.1%} confidence)

Please write complete meeting minutes in markdown.

Transcript:
{transcription}
"""

In [ ]:
messages = [
    {"role": "system", "content": system_message},
    {"role": "user",   "content": user_prompt}
]

In [ ]:
def generator(model_key, messages):
  config = MODELS[model_key]
  model_id = config["id"]
  quant = config["quant"]
  tokens = config["tokens"]

  print(f"Model - {model_id}")
  print(f"Quantization - {'yes' if quant else 'no'}")
  print(f"Max tokens - {tokens}")

  #tokenizer
  tokenizer = AutoTokenizer.from_pretrained(model_id)

  # Llama has no pad_token defined by default
  # use <|eot_id|> as placeholder for empty positions
  tokenizer.pad_token = tokenizer.eos_token

  input_ids = tokenizer.apply_chat_template(
      messages,
      return_tensors = "pt",
      add_generation_prompt = True
  ).to("cuda")

  # input ids are list of numbers representing the prompt
  # attention_mask is same as input_ids -
  # creates a tensor of all 1s no 0s because we are handling one prompt at a time
  attention_mask = torch.ones_like(input_ids['input_ids'], dtype = torch.long, device = "cuda")

  # shape[1] gets the number of tokens.
  token_count = input_ids['input_ids'].shape[1]
  print(f"Input tokens: {token_count:,} (your prompt converted to {token_count:,} numbers)")

  if quant:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            quantization_config=quant_config
        )
  else:
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            torch_dtype=torch.float16
        )

  memory_mb = model.get_memory_footprint() / 1e6
  print(f"Model loaded. Memory footprint: {memory_mb:,.0f} MB")

    # TextStreamer prints each token as it's decoded
    # Without it: you wait for ALL tokens, then see everything at once
    # With it: tokens appear one by one (like ChatGPT streaming)
  streamer = TextStreamer(tokenizer)

    # model.generate() is the core inference call
    # It runs the autoregressive loop:
    # input → predict next token → append → predict next → ... × max_new_tokens
  outputs = model.generate(
        input_ids=input_ids['input_ids'],
        attention_mask=attention_mask,
        max_new_tokens=tokens,
        streamer=streamer
  )

    # outputs[0] = full sequence (input tokens + generated tokens)
    # tokenizer.decode() converts token IDs → readable string
    # skip_special_tokens=True → removes <|eot_id|> etc. from output

  full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

  response = full_output
  for marker in ["assistant\n", "Assistant:\n", "<|assistant|>\n"]:
      if marker in full_output:
          response = full_output.split(marker)[-1].strip()
          break
  del model, input_ids, attention_mask, outputs
  gc.collect()
  torch.cuda.empty_cache()
  print(f"\n GPU memory freed. Ready for next model.")

  return response


In [ ]:
#run the models and compare

results = {}
models_to_run = [
    "gemma",      # smallest, fastest (270M params, no quantization)
    "llama",      # Meta's model (1B params, 4-bit quantized)
    "phi",
]

for model_key in models_to_run:
    results[model_key] = generator(model_key, messages)
